## PakMart Traders – Data Discovery (EDA)

Exploratory data analysis on the Pakistani retail sales data.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
sns.set_theme(style='darkgrid', palette='deep', rc={'figure.figsize': (9, 6)})

### Load fact_sale and join dimensions

In [ ]:
SEED = 42

fact_sale    = spark.table('fact_sale')
dim_city     = spark.table('dimension_city')
dim_employee = spark.table('dimension_employee')
dim_stock    = spark.table('dimension_stock_item')
dim_date     = spark.table('dimension_date')
dim_customer = spark.table('dimension_customer')

# Sample for EDA (0.5% of fact rows)
sample_df = fact_sale.sample(True, 0.005, seed=SEED)
print(f'Sample size: {sample_df.count():,}')

In [ ]:
display(fact_sale.summary())

### Sales by Province (StateProvince)

In [ ]:
from pyspark.sql.functions import sum as _sum, col

sales_by_province = fact_sale \
    .join(dim_city, 'CityKey') \
    .groupBy('StateProvince') \
    .agg(_sum('TotalIncludingTax').alias('TotalSalesPKR')) \
    .orderBy('TotalSalesPKR', ascending=False)

province_pd = sales_by_province.toPandas()

plt.figure(figsize=(10, 6))
sns.barplot(data=province_pd, x='TotalSalesPKR', y='StateProvince', palette='Blues_r')
plt.title('Total Sales by Province (PKR)')
plt.xlabel('Total Sales (PKR)')
plt.ylabel('Province')
plt.tight_layout()
plt.show()

### Monthly Revenue Trend – Pakistani Fiscal Calendar

Ramadan and Eid peaks should be visible.

In [ ]:
monthly_sales = fact_sale \
    .join(dim_date, fact_sale.InvoiceDateKey == dim_date.Date) \
    .groupBy('CalendarYear', 'CalendarMonthNumber', 'ShortMonth') \
    .agg(_sum('TotalIncludingTax').alias('MonthlySales')) \
    .orderBy('CalendarYear', 'CalendarMonthNumber')

monthly_pd = monthly_sales.toPandas()
monthly_pd['YearMonth'] = monthly_pd['CalendarYear'].astype(str) + '-' + monthly_pd['ShortMonth']

plt.figure(figsize=(16, 5))
plt.plot(range(len(monthly_pd)), monthly_pd['MonthlySales'], marker='o', linewidth=1.5)
plt.xticks(range(len(monthly_pd)), monthly_pd['YearMonth'], rotation=90, fontsize=7)
plt.title('Monthly Sales Trend (2019-2022) — PakMart Traders (PKR)')
plt.ylabel('Total Sales (PKR)')
plt.tight_layout()
plt.show()

### Top 10 Products by Revenue

In [ ]:
top_products = fact_sale \
    .join(dim_stock, 'StockItemKey') \
    .groupBy('StockItem') \
    .agg(_sum('TotalIncludingTax').alias('Revenue'), _sum('Quantity').alias('UnitsSold')) \
    .orderBy('Revenue', ascending=False) \
    .limit(10)

top_pd = top_products.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(ax=axes[0], data=top_pd, x='Revenue',   y='StockItem', palette='Greens_r').set_title('Top 10 by Revenue (PKR)')
sns.barplot(ax=axes[1], data=top_pd, x='UnitsSold', y='StockItem', palette='Oranges_r').set_title('Top 10 by Units Sold')
plt.tight_layout()
plt.show()

### Correlation heatmap – numeric sales columns

In [ ]:
corr_pd = sample_df.select(
    'Quantity','UnitPrice','TaxRate','TotalExcludingTax','TaxAmount','Profit','TotalIncludingTax','TotalDryItems','TotalChillerItems'
).toPandas()

corr_pd[['Quantity','UnitPrice','TaxRate','TotalExcludingTax','TaxAmount','Profit','TotalIncludingTax']] = \
    corr_pd[['Quantity','UnitPrice','TaxRate','TotalExcludingTax','TaxAmount','Profit','TotalIncludingTax']].astype(float)

plt.figure(figsize=(10, 7))
sns.heatmap(corr_pd.corr(), annot=True, fmt='.3f', cmap='Greens')
plt.title('Correlation Heatmap – PakMart Sales Metrics')
plt.tight_layout()
plt.show()

### Profit margin by Sales Territory

In [ ]:
from pyspark.sql.functions import round as _round

margin_df = fact_sale.join(dim_city, 'CityKey') \
    .groupBy('SalesTerritory') \
    .agg(
        _sum('Profit').alias('TotalProfit'),
        _sum('TotalExcludingTax').alias('TotalRevenue')
    )

from pyspark.sql.functions import expr
margin_df = margin_df.withColumn('ProfitMarginPct', _round(col('TotalProfit') / col('TotalRevenue') * 100, 2))
margin_pd = margin_df.orderBy('ProfitMarginPct', ascending=False).toPandas()

sns.barplot(data=margin_pd, x='ProfitMarginPct', y='SalesTerritory', palette='Blues_d')
plt.title('Profit Margin % by Sales Territory')
plt.xlabel('Profit Margin (%)')
plt.tight_layout()
plt.show()